# PySpark Hands-on: Structured Streaming

---

> **Datasets:** generados en vivo dentro del notebook. Usamos `rate` source de Spark (genera filas automaticamente) y simulacion de Kafka con archivos JSON escritos por un thread auxiliar.

> **Advertencia importante:** las queries streaming son **asincronas**. Cada `writeStream.start()` devuelve un objeto `query` que sigue corriendo en segundo plano. Siempre hay que **detenerlo** con `query.stop()` antes de seguir. Si no, se acumulan streams y Colab se vuelve lento.


---
## Fundamentos de Structured Streaming

### Setup


In [1]:
import os
import sys
os.environ.pop("SPARK_HOME", None)
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["HADOOP_HOME"] = r"C:\hadoop"

In [2]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("Clase4")
    .master("local[*]")
    .config("spark.driver.host", "127.0.0.1")
    .config("spark.driver.bindAddress", "127.0.0.1")
    .config("spark.sql.streaming.statefulOperator.checkCorrectness.enabled", "false") # No hagas esa revisión estricta. Spark normalmente revisa si tus cálculos podrían estar mal
    .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")
print("SparkSession lista. Version:", spark.version)

SparkSession lista. Version: 4.1.2


### Batch vs Streaming: el cambio mental

Hasta ahora todo lo que hicimos fue **batch**: leer un dataset completo, procesarlo, escribir resultado, terminar.

En **streaming**, los datos llegan **de a poco, todo el tiempo, sin que sepamos cuando va a terminar**. Spark resuelve esto con un truco genial: **micro-batches**.

**Idea de Structured Streaming:**

1. Cada N segundos, Spark agarra los datos nuevos que llegaron desde el ultimo procesamiento.
2. Los procesa como si fueran un mini-batch.
3. Actualiza el resultado.
4. Repite.

Lo poderoso: **el codigo es casi identico al batch**. Lo unico que cambia es `read` -> `readStream` y `write` -> `writeStream`. Todas las transformaciones (`select`, `filter`, `groupBy`, `join`) funcionan igual.

**Casos de uso reales:**

- Deteccion de fraude en tiempo real (transacciones bancarias).
- Dashboards en vivo (numero de visitas al sitio en los ultimos 5 min).
- Alertas de monitoreo (si CPU > 90% por 3 min seguidos, avisar).
- Procesamiento de logs (IPs sospechosas, errores 500).
- Recomendaciones personalizadas que reaccionan al ultimo click.


### Primer stream: rate source

`rate` es la fuente mas simple: **genera filas automaticamente** con `(timestamp, value)`. Sirve para aprender sin tener que configurar nada externo.

Cada fila tiene:
- `timestamp` (cuando se genero la fila).
- `value` (un long autoincremental: 0, 1, 2, 3...).

Vamos a generar 5 filas por segundo.


In [ ]:
# Definir el stream (todavia no arranca)
df_rate = (
    spark.readStream
    .format("rate")
    .option("rowsPerSecond", 5)
    .load()
)

# isStreaming nos confirma que es un stream, no un DataFrame estatico
print("Es streaming?:", df_rate.isStreaming)
df_rate.printSchema()

**Notar lo importante:** `df_rate` es un DataFrame igual que cualquier otro, **pero todavia no hay datos**. Recien cuando hagamos `writeStream.start()`, Spark va a empezar a generar filas y aplicarles las transformaciones.

Vamos a aplicarle una transformacion (multiplicar el value por 10) y escribirlo a un **memory sink** (una tabla en memoria que podemos consultar con SQL).


In [ ]:
from pyspark.sql.functions import col

df_transformado = df_rate.withColumn("value_x10", col("value") * 10)

# Lanzar el stream a memoria
query_rate = (
    df_transformado.writeStream
    .format("memory")
    .queryName("tabla_rate")
    .outputMode("append")
    .start()
)

print("Stream activo. ID:", query_rate.id)
print("Activo?:", query_rate.isActive)

El stream ya esta corriendo en segundo plano. Esperemos 10 segundos y consultemos la tabla en memoria.

In [ ]:
import time

# Esperamos 10 segundos a que se acumulen filas
time.sleep(10)

# Consultar la tabla en memoria con SQL
spark.sql("SELECT * FROM tabla_rate ORDER BY value DESC LIMIT 10").show(truncate=False)
print("Total filas acumuladas:", spark.sql("SELECT COUNT(*) FROM tabla_rate").collect()[0][0])

**IMPORTANTE: detener el stream antes de seguir.** Si no, sigue corriendo y consume CPU.

In [ ]:
query_rate.stop()
print("Stream detenido. Activo?:", query_rate.isActive)

### File source: simulando Kafka con archivos

En produccion los streams casi siempre vienen de **Kafka** o **Kinesis**. Pero en Colab no podemos correr Kafka. La buena noticia: Spark Streaming tiene un **file source** que mira una carpeta y **procesa cada archivo nuevo** que aparece.

Vamos a montar un escenario realista:

1. Un **thread productor** que escribe un archivo JSON nuevo cada 2 segundos (simulando un sistema que envia eventos a una carpeta compartida).
2. Un **stream consumidor** que lee esos archivos y procesa los eventos.

Es el patron tipico de un pipeline de ingesta basado en archivos (muy comun antes de Kafka).


In [ ]:
# Limpieza por si quedo algo de corridas anteriores /¿Esta carpeta existe? /Si existe, la borra completa.
import shutil, os
for d in ["/tmp/eventos_in", "/tmp/eventos_chk"]:
    if os.path.exists(d):
        shutil.rmtree(d)
os.makedirs("/tmp/eventos_in", exist_ok=True)
print("Carpetas limpias")

**Productor (thread auxiliar)**: escribe un archivo JSON con varias transacciones cada 2 segundos. Le ponemos un limite para que no corra para siempre.

In [ ]:
import json, time, threading, random
from datetime import datetime

SUCURSALES = ["Vespucio", "Maipu", "Las_Condes", "Concepcion", "Antofagasta"]
PRODUCTOS  = ["leche", "yogurt", "queso", "mantequilla", "crema"]

def productor(carpeta, segundos_total=60, intervalo=2):
    """Escribe un JSON con 3 a 8 transacciones cada `intervalo` segundos."""
    fin = time.time() + segundos_total
    contador = 0
    while time.time() < fin:
        eventos = []
        for _ in range(random.randint(3, 8)):
            eventos.append({
                "id_tx":      contador,
                "timestamp":  datetime.utcnow().isoformat(),
                "sucursal":   random.choice(SUCURSALES),
                "producto":   random.choice(PRODUCTOS),
                "cantidad":   random.randint(1, 20),
                "monto":      round(random.uniform(500, 25000), 0),
            })
            contador += 1

        # Un archivo JSON Lines por tick
        archivo = f"{carpeta}/tx_{int(time.time()*1000)}.jsonl"
        with open(archivo, "w") as f:
            for e in eventos:
                f.write(json.dumps(e) + "\n")

        time.sleep(intervalo)

print("Productor definido (todavia no esta corriendo).")

**Schema del stream**: lo definimos a mano. Esto es **obligatorio** con file source en streaming (Spark no puede hacer `inferSchema` porque tendria que escanear archivos futuros).

In [ ]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, DoubleType, TimestampType

schema_eventos = StructType([
    StructField("id_tx",     IntegerType(),   True),
    StructField("timestamp", TimestampType(), True),
    StructField("sucursal",  StringType(),    True),
    StructField("producto",  StringType(),    True),
    StructField("cantidad",  IntegerType(),   True),
    StructField("monto",     DoubleType(),    True),
])
print("Schema definido")

**Definir el stream** sobre la carpeta. `maxFilesPerTrigger` limita cuantos archivos procesa por micro-batch (sin esto Spark agarra todos los archivos disponibles, lo que se vuelve un batch normal).

In [ ]:
df_stream = (
    spark.readStream
    .schema(schema_eventos)
    .option("maxFilesPerTrigger", 1)        # 1 archivo nuevo por micro-batch
    .json("/tmp/eventos_in")
)

print("Es streaming?:", df_stream.isStreaming)
df_stream.printSchema()

**Lanzar el consumidor** primero (a memoria, para poder inspeccionar), **y despues** arrancar el productor en un thread.

In [ ]:
query_eventos = (
    df_stream.writeStream
    .format("memory")
    .queryName("eventos_memoria")
    .outputMode("append")
    .start()
)

# Arrancar el productor en un thread (corre 30 segundos)
t = threading.Thread(target=productor, args=("/tmp/eventos_in", 30, 2), daemon=True)
t.start()

print("Productor y consumidor activos. Esperando 15 segundos...")
time.sleep(15)

# Consultar lo que se haya acumulado
spark.sql("SELECT * FROM eventos_memoria ORDER BY id_tx DESC LIMIT 10").show(truncate=False)
total = spark.sql("SELECT COUNT(*) FROM eventos_memoria").collect()[0][0]
print(f"Eventos procesados hasta ahora: {total}")

Esperemos un poco mas y veamos como sigue creciendo.

In [ ]:
time.sleep(15)

total = spark.sql("SELECT COUNT(*) FROM eventos_memoria").collect()[0][0]
print(f"Eventos procesados ahora: {total}")
spark.sql("SELECT sucursal, COUNT(*) AS cantidad FROM eventos_memoria GROUP BY sucursal ORDER BY cantidad DESC").show()

In [ ]:
# Detener el stream
query_eventos.stop()
print("Stream detenido. Activo?:", query_eventos.isActive)

### Output modes: append, complete, update

Hay 3 modos de salida:

| Modo | Que escribe | Cuando usarlo |
|------|-------------|---------------|
| **`append`** | Solo filas **nuevas** que llegaron en este batch | Streams sin agregacion. Default seguro. |
| **`complete`** | **Toda la tabla resultado** cada vez | Agregaciones (`groupBy.count()`, totales globales). Requiere agregacion. |
| **`update`** | Solo las filas **modificadas** desde el ultimo batch | Agregaciones con keys (top N por categoria). Mas eficiente que complete. |

**Reglas duras:**
- Con agregacion sin watermark: solo `complete` o `update`.
- Sin agregacion: solo `append`.
- Con join stream-static: cualquiera.

Veamos `complete` mode con un conteo en vivo.


### Demo: conteo en vivo con `complete` mode

Vamos a contar transacciones por sucursal en vivo. Cada vez que llega un archivo nuevo, el conteo se actualiza.


In [ ]:
# Limpiamos la carpeta y volvemos a arrancar el productor
shutil.rmtree("/tmp/eventos_in", ignore_errors=True)
os.makedirs("/tmp/eventos_in", exist_ok=True)

df_stream2 = (
    spark.readStream
    .schema(schema_eventos)
    .option("maxFilesPerTrigger", 1)
    .json("/tmp/eventos_in")
)

# Agregacion: conteo por sucursal
df_conteo = df_stream2.groupBy("sucursal").count()

# Output mode complete: toda la tabla cada vez
query_conteo = (
    df_conteo.writeStream
    .format("memory")
    .queryName("conteo_sucursales")
    .outputMode("complete")              # toda la tabla cada vez
    .start()
)

# Arrancar productor en thread
t2 = threading.Thread(target=productor, args=("/tmp/eventos_in", 90, 2), daemon=True)
t2.start()

In [ ]:
print("Stream agregado corriendo. Espera 15 segundos...")
time.sleep(15)
spark.sql("SELECT * FROM conteo_sucursales ORDER BY count DESC").show()

In [ ]:
# Esperamos otros 10 segundos y vemos como crecio
time.sleep(10)
spark.sql("SELECT * FROM conteo_sucursales ORDER BY count DESC").show()

In [ ]:
query_conteo.stop()
print("Stream detenido")

### Checkpoints: que pasa si se cae el job

En produccion los streams corren **dias o semanas seguidos**. Si el proceso se cae (memoria, red, deploy), tiene que poder **retomar desde donde se quedo**, no desde cero.

Para eso esta el **checkpoint**: una carpeta donde Spark guarda el estado del stream (que archivos ya proceso, posicion en el topic de Kafka, estado de agregaciones, etc.).

**Regla en produccion:** TODO writeStream tiene `checkpointLocation`. Sin excepciones.

Veamos cuando se usa de verdad: con sink Parquet (no se puede usar memory sink con persistencia real).


In [ ]:
# Limpiar
shutil.rmtree("/tmp/eventos_in", ignore_errors=True)
shutil.rmtree("/tmp/eventos_out", ignore_errors=True)
shutil.rmtree("/tmp/eventos_chk", ignore_errors=True)
os.makedirs("/tmp/eventos_in", exist_ok=True)

# Stream a parquet, CON checkpoint
df_stream3 = (
    spark.readStream
    .schema(schema_eventos)
    .option("maxFilesPerTrigger", 1)
    .json("/tmp/eventos_in")
)

query_parquet = (
    df_stream3.writeStream
    .format("parquet")
    .option("path",               "/tmp/eventos_out")
    .option("checkpointLocation", "/tmp/eventos_chk")   # critico
    .outputMode("append")
    .start()
)

# Productor
t3 = threading.Thread(target=productor, args=("/tmp/eventos_in", 15, 2), daemon=True)
t3.start()

print("Stream + checkpoint activo. Esperando...")
time.sleep(18)
query_parquet.stop()

# Que escribio?
print("--- Parquet generado ---")
!ls /tmp/eventos_out/
print("--- Carpeta de checkpoint ---")
!ls /tmp/eventos_chk/

La carpeta `_chk` tiene archivos de metadata internos de Spark: `commits/`, `offsets/`, `state/`. Si el job se cae y arrancas con el **mismo checkpointLocation**, Spark retoma exactamente donde estaba.

**Importante:** si cambias el codigo (por ejemplo agregas una columna), a veces tienes que **borrar el checkpoint** porque el schema no calza. Es la fuente N1 de bugs en streaming.


In [ ]:
# Leer el parquet generado para verificar
df_persistido = spark.read.parquet("/tmp/eventos_out")
print(f"Eventos persistidos: {df_persistido.count()}")
df_persistido.show(5)

---
## Agregaciones, ventanas temporales y watermarks

### Ventanas temporales: la cosa mas importante de streaming

En batch agrupamos por columnas categoricas (sucursal, producto). En streaming agrupamos sobre todo por **tiempo**: "ultimos 5 minutos", "cada hora", "ventanas de 1 dia con paso de 6 horas".

Spark tiene 3 tipos de ventana temporal:

| Tipo | Como se ven | Caso de uso |
|------|-------------|-------------|
| **Tumbling** | Bloques fijos sin solapamiento: [0-5m], [5-10m], [10-15m]... | KPIs por intervalo (visitas por hora). |
| **Sliding** | Bloques fijos que se solapan: cada 1 min, ventana de 5 min | Promedios moviles, alertas. |
| **Session** | Tamano variable, agrupa eventos cercanos | Sesiones de usuario en web. |

La sintaxis basica:

```python
from pyspark.sql.functions import window
df.groupBy(window(col("timestamp"), "5 minutes"))   # tumbling
df.groupBy(window(col("timestamp"), "5 minutes", "1 minute"))  # sliding
```

Vamos a ver un ejemplo concreto.


### Tumbling window: ingreso por minuto

Limpiamos y arrancamos un nuevo stream. Vamos a calcular el monto total por **ventana de 30 segundos** (tumbling).


In [ ]:
shutil.rmtree("/tmp/eventos_in", ignore_errors=True)
os.makedirs("/tmp/eventos_in", exist_ok=True)

df_stream_w = (
    spark.readStream
    .schema(schema_eventos)
    .option("maxFilesPerTrigger", 1)
    .json("/tmp/eventos_in")
)

# Inspeccionar: que columnas tiene? Necesitamos `timestamp` para ventana
df_stream_w.printSchema()

In [ ]:
from pyspark.sql.functions import window, sum as f_sum, count

# Ventana tumbling de 30 segundos sobre la columna timestamp
df_ventana = (
    df_stream_w
    .groupBy(window(col("timestamp"), "30 seconds"))
    .agg(
        count("*").alias("transacciones"),
        f_sum("monto").alias("monto_total")
    )
)

query_ventana = (
    df_ventana.writeStream
    .format("memory")
    .queryName("ventanas_tumbling")
    .outputMode("complete")
    .start()
)

# Productor
t4 = threading.Thread(target=productor, args=("/tmp/eventos_in", 60, 2), daemon=True)
t4.start()

print("Stream con ventana corriendo. Esperando 30 segundos...")
time.sleep(30)
spark.sql("""
    SELECT
        window.start AS desde,
        window.end   AS hasta,
        transacciones,
        ROUND(monto_total, 0) AS monto_total
    FROM ventanas_tumbling
    ORDER BY window.start
""").show(truncate=False)

In [ ]:
# Esperamos otros 30 segundos
time.sleep(30)
spark.sql("""
    SELECT
        window.start AS desde,
        window.end   AS hasta,
        transacciones,
        ROUND(monto_total, 0) AS monto_total
    FROM ventanas_tumbling
    ORDER BY window.start
""").show(truncate=False)
query_ventana.stop()

**Lo importante de la salida:** la columna `window` es un **struct** con `start` y `end`. Cada fila representa **una ventana cerrada** (intervalo de 30 segundos) con la agregacion adentro.

Notar que los inicios estan alineados a multiplos exactos de 30s (`:00`, `:30`, `:00`...) sin importar cuando arrancamos el stream. Spark redondea al limite logico.


### Sliding window: monto promedio movil

Sliding window: la ventana es de 1 minuto, pero **se desliza cada 20 segundos**. Asi tenemos un valor actualizado mas frecuentemente (cada 20s en vez de cada minuto).


In [ ]:
shutil.rmtree("/tmp/eventos_in", ignore_errors=True)
os.makedirs("/tmp/eventos_in", exist_ok=True)

from pyspark.sql.functions import avg

df_stream_s = (
    spark.readStream
    .schema(schema_eventos)
    .option("maxFilesPerTrigger", 1)
    .json("/tmp/eventos_in")
)

# Sliding window: 1 min de ventana, paso de 20 segundos
df_sliding = (
    df_stream_s
    .groupBy(window(col("timestamp"), "1 minute", "20 seconds"))
    .agg(
        count("*").alias("transacciones"),
        avg("monto").alias("monto_promedio")
    )
)

query_sliding = (
    df_sliding.writeStream
    .format("memory")
    .queryName("ventanas_sliding")
    .outputMode("complete")
    .start()
)

t5 = threading.Thread(target=productor, args=("/tmp/eventos_in", 60, 2), daemon=True)
t5.start()

print("Sliding window corriendo. Esperando 60 segundos...")
time.sleep(60)
spark.sql("""
    SELECT
        window.start AS desde,
        window.end   AS hasta,
        transacciones,
        ROUND(monto_promedio, 0) AS promedio
    FROM ventanas_sliding
    ORDER BY window.start
""").show(15, truncate=False)
query_sliding.stop()

Notar como las ventanas **se solapan**: cada ventana mide 1 minuto pero arrancan cada 20 segundos. Asi tenemos un indicador actualizado 3 veces por minuto.

**Cuando usar cada uno:**
- **Tumbling**: KPIs operacionales (visitas por hora, ventas por dia).
- **Sliding**: alertas y deteccion de tendencias (monto promedio en los ultimos 5 minutos, actualizado cada 30s).


### - Watermarks: manejar eventos tardios sin que la memoria explote

Problema real: en streaming los eventos **no siempre llegan en orden**. Un evento generado a las 10:00 puede llegar a Spark a las 10:03 (por delays de red, reintentos, buffers locales del cliente).

Si no hacemos nada, Spark guarda **todas** las ventanas pasadas en memoria por si llega un evento tardio. Despues de unas horas, la memoria explota.

**Solucion: watermark.** Le decimos a Spark: "puedes asumir que un evento mas viejo que X minutos ya no va a llegar". Despues de ese tiempo, **Spark descarta el estado de esas ventanas** y libera memoria.

```python
df.withWatermark("timestamp", "10 minutes").groupBy(window(...))
```

- Si llega un evento dentro del watermark, **se procesa**.
- Si llega despues del watermark, **se descarta** (en append mode).

Vamos a verlo con un ejemplo.


In [ ]:
import os, shutil, json, time, threading, random
from datetime import datetime, UTC

from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)

from pyspark.sql.functions import (
    col, window, count, sum as f_sum,
    to_timestamp
)

# -------------------------------------------------------
# 1. Detener streams anteriores
# -------------------------------------------------------
for q in spark.streams.active:
    q.stop()

# -------------------------------------------------------
# 2. Limpiar carpetas
# -------------------------------------------------------
for d in ["/tmp/eventos_in", "/tmp/eventos_chk_wm"]:
    shutil.rmtree(d, ignore_errors=True)

os.makedirs("/tmp/eventos_in", exist_ok=True)

# -------------------------------------------------------
# 3. Schema: timestamp entra como texto
#    Luego lo convertimos explícitamente a timestamp
# -------------------------------------------------------
schema_eventos_raw = StructType([
    StructField("id_tx", IntegerType(), True),
    StructField("timestamp", StringType(), True),
    StructField("sucursal", StringType(), True),
    StructField("producto", StringType(), True),
    StructField("cantidad", IntegerType(), True),
    StructField("monto", DoubleType(), True),
])

# -------------------------------------------------------
# 4. Productor
# -------------------------------------------------------
SUCURSALES = ["Vespucio", "Maipu", "Las_Condes", "Concepcion", "Antofagasta"]
PRODUCTOS  = ["leche", "yogurt", "queso", "mantequilla", "crema"]

def productor(carpeta, segundos_total=90, intervalo=2):
    fin = time.time() + segundos_total
    contador = 0

    while time.time() < fin:
        eventos = []

        for _ in range(random.randint(3, 8)):
            eventos.append({
                "id_tx": contador,
                "timestamp": datetime.now(UTC).isoformat(),
                "sucursal": random.choice(SUCURSALES),
                "producto": random.choice(PRODUCTOS),
                "cantidad": random.randint(1, 20),
                "monto": float(round(random.uniform(500, 25000), 0)),
            })
            contador += 1

        archivo = f"{carpeta}/tx_{int(time.time() * 1000)}.jsonl"

        with open(archivo, "w") as f:
            for e in eventos:
                f.write(json.dumps(e) + "\n")

        time.sleep(intervalo)

# -------------------------------------------------------
# 5. Leer stream
# -------------------------------------------------------
df_stream_wm_raw = (
    spark.readStream
    .schema(schema_eventos_raw)
    .option("maxFilesPerTrigger", 1)
    .json("/tmp/eventos_in")
)

# Convertimos timestamp string a TimestampType
df_stream_wm = (
    df_stream_wm_raw
    .withColumn("event_time", to_timestamp(col("timestamp")))
    .drop("timestamp")
)

# -------------------------------------------------------
# 6. Agregación con watermark
# -------------------------------------------------------
df_con_watermark = (
    df_stream_wm
    .withWatermark("event_time", "60 seconds")
    .groupBy(window(col("event_time"), "30 seconds"))
    .agg(
        count("*").alias("transacciones"),
        f_sum("monto").alias("monto_total")
    )
)

# -------------------------------------------------------
# 7. Escribir en memoria con append
# -------------------------------------------------------
query_wm = (
    df_con_watermark.writeStream
    .format("memory")
    .queryName("ventanas_wm")
    .outputMode("append")
    .option("checkpointLocation", "/tmp/eventos_chk_wm")
    .trigger(processingTime="5 seconds")
    .start()
)

# -------------------------------------------------------
# 8. Iniciar productor
# -------------------------------------------------------
t6 = threading.Thread(
    target=productor,
    args=("/tmp/eventos_in", 90, 2),
    daemon=True
)
t6.start()

print("Stream con watermark corriendo. Esperando 90 segundos...")
time.sleep(90)

# Forzar procesamiento de archivos pendientes
query_wm.processAllAvailable()

# -------------------------------------------------------
# 9. Consultar resultado
# -------------------------------------------------------
spark.sql("""
    SELECT
        window.start AS desde,
        window.end   AS hasta,
        transacciones,
        ROUND(monto_total, 0) AS monto_total
    FROM ventanas_wm
    ORDER BY desde
""").show(15, truncate=False)

query_wm.stop()

Notar dos cosas:

1. **Las ventanas mas recientes no aparecen todavia**: Spark las tiene "en proceso" y solo las emite cuando el watermark las marca como cerradas (ningun evento mas viejo va a llegar).
2. **Append mode funciona con agregacion** SOLO si hay watermark. Sin watermark seria `complete` o `update`.

**Como elegir el watermark:**
- Muy chico (1 segundo): muchas ventanas tardias se descartan -> perdes datos.
- Muy grande (1 hora): la memoria sigue creciendo demasiado -> caro.
- En la practica: entre 1 y 30 minutos segun el negocio.


### - Ejercicio guiado: top productos por ventana

**Pregunta:** queremos saber **cual es el producto mas vendido en ventanas tumbling de 30 segundos**.


Pero ojo: en streaming **no se puede `orderBy` global** (Spark no sabe el "fin" de los datos). Hay dos opciones:
- Calcular el agregado y leerlo desde la tabla en memoria con SQL para ordenar (lo que vamos a hacer).
- Usar `foreachBatch` para hacer post-procesamiento por batch (lo vemos en el bloque 3).


In [ ]:
shutil.rmtree("/tmp/eventos_in", ignore_errors=True)
os.makedirs("/tmp/eventos_in", exist_ok=True)

df_stream_e = (
    spark.readStream
    .schema(schema_eventos)
    .option("maxFilesPerTrigger", 1)
    .json("/tmp/eventos_in")
)

df_top_productos = (
    df_stream_e
    .withWatermark("timestamp", "1 minute")
    .groupBy(window(col("timestamp"), "30 seconds"), "producto")
    .agg(f_sum("cantidad").alias("unidades"))
)

query_top = (
    df_top_productos.writeStream
    .format("memory")
    .queryName("top_productos")
    .outputMode("update")
    .start()
)

t7 = threading.Thread(target=productor, args=("/tmp/eventos_in", 60, 2), daemon=True)
t7.start()

print("Stream corriendo. Esperando 45 segundos...")
time.sleep(45)

# El orderBy lo hacemos sobre la tabla en memoria, no sobre el stream
spark.sql("""
    SELECT
        window.start AS desde,
        producto,
        unidades
    FROM top_productos
    WHERE window.start = (SELECT MAX(window.start) FROM top_productos)
    ORDER BY unidades DESC
""").show(truncate=False)

query_top.stop()

---
## Joins, foreachBatch y caso integrador
### Stream-static join: enriquecer eventos con datos de referencia

Caso clasico: el stream trae **solo IDs** (eficiente para transportar), y necesitamos enriquecerlos con nombres, categorias, precios, etc. que estan en una **tabla estatica** (cargada una vez).

Spark deja hacer **join entre un stream y un DataFrame estatico** sin problema. Es el patron mas usado en streaming real.


In [ ]:
# Cargamos un catalogo estatico de productos
from pyspark.sql import Row

productos_data = [
    Row(producto="leche",       categoria="lacteos",     precio_lista=1200),
    Row(producto="yogurt",      categoria="lacteos",     precio_lista=900),
    Row(producto="queso",       categoria="lacteos",     precio_lista=3500),
    Row(producto="mantequilla", categoria="lacteos",     precio_lista=2800),
    Row(producto="crema",       categoria="lacteos",     precio_lista=2200),
]
df_productos = spark.createDataFrame(productos_data)
df_productos.show()

In [ ]:
shutil.rmtree("/tmp/eventos_in", ignore_errors=True)
os.makedirs("/tmp/eventos_in", exist_ok=True)

df_stream_j = (
    spark.readStream
    .schema(schema_eventos)
    .option("maxFilesPerTrigger", 1)
    .json("/tmp/eventos_in")
)

# Join stream + estatico
df_enriquecido = df_stream_j.join(df_productos, on="producto", how="inner")

# Agregamos una columna "margen" calculada al vuelo
from pyspark.sql.functions import expr
df_con_margen = df_enriquecido.withColumn(
    "diferencia_pesos",
    (col("monto") / col("cantidad")) - col("precio_lista")
)

query_join = (
    df_con_margen.writeStream
    .format("memory")
    .queryName("eventos_enriquecidos")
    .outputMode("append")
    .start()
)

t8 = threading.Thread(target=productor, args=("/tmp/eventos_in", 30, 2), daemon=True)
t8.start()

print("Stream con join activo. Esperando 20 segundos...")
time.sleep(20)
spark.sql("""
    SELECT producto, categoria, cantidad, monto, ROUND(diferencia_pesos, 0) AS dif_pesos
    FROM eventos_enriquecidos
    ORDER BY ABS(diferencia_pesos) DESC
    LIMIT 10
""").show()

query_join.stop()

Notar que el join se hizo *fila por micro-batch**. El catalogo estatico se mantiene en memoria del driver y se reutiliza en cada batch.

**Limitacion importante:** stream-stream join existe pero es **mucho mas complejo** (requiere watermark en ambos lados, define rangos temporales validos, etc.). Para esta clase nos quedamos con stream-static que cubre el 95% de los casos.


### foreachBatch: hacer cualquier cosa con cada micro-batch

Hasta aca usamos sinks built-in (memory, parquet). Pero a veces necesitamos hacer cosas que ningun sink built-in soporta:

- Escribir a una base de datos relacional con `INSERT ... ON CONFLICT UPDATE`.
- Enviar alertas por email/Slack si pasa algo.
- Aplicar transformaciones que solo funcionan en batch (`orderBy` global, por ejemplo).
- Escribir a multiples destinos a la vez.

Para eso esta **`foreachBatch`**: una funcion Python que recibe **el DataFrame del batch** y un **batch_id**, y puede hacer **cualquier cosa** con ellos.


In [ ]:
shutil.rmtree("/tmp/eventos_in", ignore_errors=True)
shutil.rmtree("/tmp/alertas", ignore_errors=True)
os.makedirs("/tmp/eventos_in", exist_ok=True)

df_stream_fb = (
    spark.readStream
    .schema(schema_eventos)
    .option("maxFilesPerTrigger", 1)
    .json("/tmp/eventos_in")
)

# Funcion que se ejecuta una vez por micro-batch
def procesar_batch(df_batch, batch_id):
    n = df_batch.count()
    if n == 0:
        return
    # 1. Imprimir resumen
    monto_total = df_batch.agg(f_sum("monto")).collect()[0][0]
    print(f"[Batch {batch_id}] {n} eventos, monto total = {monto_total}")

    # 2. Filtrar transacciones grandes y guardarlas aparte
    grandes = df_batch.filter(col("monto") > 15000)
    if grandes.count() > 0:
        (grandes
            .write
            .mode("append")
            .parquet("/tmp/alertas"))
        print(f"   -> {grandes.count()} transacciones grandes guardadas en /tmp/alertas")

query_fb = (
    df_stream_fb.writeStream
    .foreachBatch(procesar_batch)
    .outputMode("append")
    .start()
)

t9 = threading.Thread(target=productor, args=("/tmp/eventos_in", 25, 2), daemon=True)
t9.start()

print("Stream con foreachBatch activo. Veras prints por cada batch:")
time.sleep(28)
query_fb.stop()

In [ ]:
# Verificamos que se hayan escrito alertas
if os.path.exists("/tmp/alertas"):
    df_alertas = spark.read.parquet("/tmp/alertas")
    print(f"Total alertas guardadas: {df_alertas.count()}")
    df_alertas.orderBy(col("monto").desc()).show(10)
else:
    print("No se generaron alertas en esta corrida")

**foreachBatch es la herramienta universal de streaming.** Si algo no se puede hacer con los sinks built-in, casi seguro se puede hacer con foreachBatch.

**Tip practico:** dentro de `procesar_batch` se pueden ejecutar tantas acciones como quieras sobre el DataFrame del batch. Es **batch normal de Spark**, no streaming, asi que valen todas las operaciones (`orderBy`, `collect`, etc.).


### - Caso integrador: detector de fraude

Vamos a poner todo lo aprendido junto. Escenario:

- Llegan transacciones bancarias a una carpeta.
- Necesitamos **detectar transacciones sospechosas** en tiempo real y mandarlas a una tabla de alertas.

**3 reglas de fraude:**

1. **Monto inusualmente alto**: cualquier transaccion > 50,000.
2. **Velocidad sospechosa**: mas de 5 transacciones del mismo cliente en 1 minuto.
3. **Hora atipica**: transaccion entre las 02:00 y las 05:00 UTC.

Construimos primero un productor que simule transacciones bancarias.


In [ ]:
# Productor de transacciones bancarias con clientes y horarios mas variados
CLIENTES = [f"cliente_{i:03d}" for i in range(1, 21)]

"""
[
    "cliente_001",
    "cliente_002",
    "cliente_003",
    ...
    "cliente_020"
]

"""

def productor_banco(carpeta, segundos_total=90, intervalo=2):
    fin = time.time() + segundos_total
    contador = 0
    while time.time() < fin:
        eventos = []
        for _ in range(random.randint(2, 6)):
            # Algunas transacciones tienen monto altisimo (5% de probabilidad)
            if random.random() < 0.05:
                monto = round(random.uniform(50000, 500000), 0)
            else:
                monto = round(random.uniform(1000, 30000), 0)

            # Algunos clientes hacen muchas en poco tiempo (sospechosos)
            if random.random() < 0.15:
                cliente = "cliente_001"     # cliente sospechoso
            else:
                cliente = random.choice(CLIENTES)

            eventos.append({
                "id_tx":     contador,
                "timestamp": datetime.utcnow().isoformat(),
                "cliente":   cliente,
                "monto":     monto,
                "tipo":      random.choice(["compra", "transferencia", "retiro_atm"]),
            })
            contador += 1

        archivo = f"{carpeta}/banco_{int(time.time()*1000)}.jsonl"
        with open(archivo, "w") as f:
            for e in eventos:
                f.write(json.dumps(e) + "\n")

        time.sleep(intervalo)

# Schema
schema_banco = StructType([
    StructField("id_tx",     IntegerType(),   True),
    StructField("timestamp", TimestampType(), True),
    StructField("cliente",   StringType(),    True),
    StructField("monto",     DoubleType(),    True),
    StructField("tipo",      StringType(),    True),
])

print("Productor de banco definido")

In [ ]:
# Limpiamos carpetas
for d in ["/tmp/banco_in", "/tmp/alertas_monto", "/tmp/alertas_velocidad", "/tmp/banco_chk1", "/tmp/banco_chk2"]:
    shutil.rmtree(d, ignore_errors=True)
os.makedirs("/tmp/banco_in", exist_ok=True)

# Stream de banco
df_banco = (
    spark.readStream
    .schema(schema_banco)
    .option("maxFilesPerTrigger", 1)
    .json("/tmp/banco_in")
)

print("Stream de banco listo")

**Regla 1: monto alto.** Es un simple `filter`. Lo escribimos a Parquet con checkpoint.

In [ ]:
# Regla 1: monto > 50,000
df_monto_alto = df_banco.filter(col("monto") > 50000).withColumn("regla", expr("'monto_alto'"))

query_r1 = (
    df_monto_alto.writeStream
    .format("parquet")
    .option("path",               "/tmp/alertas_monto")
    .option("checkpointLocation", "/tmp/banco_chk1")
    .outputMode("append")
    .start()
)

print("Detector regla 1 (monto alto) activo")

**Regla 2: velocidad sospechosa.** Mas de 5 transacciones del mismo cliente en 1 minuto. Usa agregacion con ventana y watermark.

In [ ]:
from pyspark.sql.functions import lit

# Regla 2: > 5 transacciones por cliente en ventana de 1 minuto
df_velocidad = (
    df_banco
    .withWatermark("timestamp", "2 minutes")
    .groupBy(window(col("timestamp"), "1 minute"), "cliente")
    .agg(count("*").alias("transacciones"))
    .filter(col("transacciones") > 5)
    .withColumn("regla", lit("velocidad"))
)

query_r2 = (
    df_velocidad.writeStream
    .format("parquet")
    .option("path",               "/tmp/alertas_velocidad")
    .option("checkpointLocation", "/tmp/banco_chk2")
    .outputMode("append")
    .start()
)

print("Detector regla 2 (velocidad) activo")

Arrancamos el productor y dejamos correr 90 segundos.

In [ ]:
t_banco = threading.Thread(target=productor_banco, args=("/tmp/banco_in", 90, 2), daemon=True)
t_banco.start()

print("Productor de banco activo. Esperando 90 segundos...")
time.sleep(95)

# Detener los streams
query_r1.stop()
query_r2.stop()
print("Streams detenidos")

In [ ]:
# Revisamos alertas generadas
print("=== ALERTAS POR MONTO ALTO ===")
if os.path.exists("/tmp/alertas_monto") and len(os.listdir("/tmp/alertas_monto")) > 1:
    df_a1 = spark.read.parquet("/tmp/alertas_monto")
    print(f"Total: {df_a1.count()}")
    df_a1.orderBy(col("monto").desc()).show(10)
else:
    print("Sin alertas (puede pasar si la corrida fue muy corta o sin transacciones grandes)")

In [ ]:
print("=== ALERTAS POR VELOCIDAD SOSPECHOSA ===")
if os.path.exists("/tmp/alertas_velocidad") and len(os.listdir("/tmp/alertas_velocidad")) > 1:
    df_a2 = spark.read.parquet("/tmp/alertas_velocidad")
    print(f"Total: {df_a2.count()}")
    df_a2.show(10, truncate=False)
else:
    print("Sin alertas de velocidad en esta corrida")

**Tarea opcional:**
- Implementar una **tercera regla de fraude** (hora atipica entre 02:00 y 05:00 UTC) y agregar como tercer stream.
- Combinar las 3 alertas en una sola tabla y enviarlas con `foreachBatch` a un Parquet unificado con la columna `regla` que identifique el tipo.
- Bajar `maxFilesPerTrigger` a 2 archivos y comparar la latencia.


In [ ]:
# Limpieza final
queries_activas = [q for q in spark.streams.active]
print(f"Queries activas: {len(queries_activas)}")
for q in queries_activas:
    q.stop()
    print(f"  Detenida: {q.id}")

spark.stop()
print("SparkSession cerrada")